# 02b — Directional wavelets

Fourier modes identify a scale and direction but are spread over the whole box. A wavelet keeps the frequency selection and localizes the response in space. Here we build one directional Morlet wavelet and compute first-order scattering coefficients.

In [ ]:
from pathlib import Path
import numpy as np
import sys, os
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

ROOT = Path(os.path.dirname(os.path.abspath('.'))).parent

OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

def savefig(fig, name):
    fig.text(0.995, 0.005, "analytic-fixture", ha="right", va="bottom",
             fontsize=7, color="#626C78")
    fig.savefig(OUTPUT_DIR / name, bbox_inches="tight")

## A simple periodic field

The field contains two localized wave packets with different wavelengths and wavevector directions. This is a controlled two-dimensional example, not a simulated universe. The angle $\alpha$ labels the wavevector; the visible ridges are perpendicular to it.

In [ ]:
n, box_size = 128, 400.0
dx = box_size / n
x = np.arange(n) * dx
X, Y = np.meshgrid(x, x, indexing="ij")

def wave_packet(center, width, mode, angle, amplitude):
    offset_x = (X - center[0] + box_size / 2) % box_size - box_size / 2
    offset_y = (Y - center[1] + box_size / 2) % box_size - box_size / 2
    along_wavevector = (
        offset_x * np.cos(angle) + offset_y * np.sin(angle)
    )
    envelope = np.exp(
        -(offset_x**2 + offset_y**2) / (2 * width**2)
    )
    return amplitude * envelope * np.cos(
        2 * np.pi * mode * along_wavevector / box_size
    )

field = wave_packet(
    center=(0.28 * box_size, 0.35 * box_size),
    width=0.10 * box_size,
    mode=8,
    angle=0.0,
    amplitude=0.70,
)
field += wave_packet(
    center=(0.70 * box_size, 0.65 * box_size),
    width=0.14 * box_size,
    mode=4,
    angle=np.pi / 4,
    amplitude=0.80,
)
field -= field.mean()

k_axis = 2 * np.pi * np.fft.fftfreq(n, d=dx)
kx, ky = np.meshgrid(k_axis, k_axis, indexing="ij")

# k0(j) is dyadic. Eight axial angles on [0, pi) give 4 x 8 channels.
central_modes = np.array([16, 8, 4, 2])
central_k = 2 * np.pi * central_modes / box_size
angles = np.arange(8) * np.pi / 8

## One complex Morlet wavelet

For $\mathbf e_\alpha=(\cos\alpha,\sin\alpha)$, use

$$
\widehat\psi_{k_0,\alpha}(\mathbf k)=
\exp\left[-\frac{|\mathbf k-k_0\mathbf e_\alpha|^2}{2\sigma_k^2}\right]
-
\exp\left[-\frac{k_0^2}{2\sigma_k^2}\right]
\exp\left[-\frac{|\mathbf k|^2}{2\sigma_k^2}\right].
$$

The second term makes $\widehat\psi(\mathbf 0)=0$, so the wavelet has zero mean. We normalize each filter so $\sum_{\mathbf k}|\widehat\psi(\mathbf k)|^2=1$. The central wavenumber is $k_0$; its real-space size is roughly proportional to $1/k_0$.

In [ ]:
def morlet_filter(k0, angle):
    sigma_k = k0 / 3
    k_parallel = kx * np.cos(angle) + ky * np.sin(angle)
    k_perpendicular = -kx * np.sin(angle) + ky * np.cos(angle)

    # TODO 1: implement the zero-mean Morlet filter from the equation above,
    # set its zero mode to zero, and normalize its discrete L2 norm to one.
    raise NotImplementedError



example_k0 = central_k[1]
example_angle = 0.0
example_filter = morlet_filter(example_k0, example_angle)
spatial_wavelet = np.fft.fftshift(
    np.fft.ifft2(example_filter)
).real
fourier_wavelet = np.fft.fftshift(np.abs(example_filter))

centered_x = (np.arange(n) - n // 2) * dx
centered_k = np.fft.fftshift(k_axis)
x_mask = np.abs(centered_x) <= 80
k_mask = np.abs(centered_k) <= 0.35

fig, axes = plt.subplots(
    1, 2, figsize=(9.6, 3.8), constrained_layout=True
)
axes[0].imshow(
    spatial_wavelet[np.ix_(x_mask, x_mask)].T,
    origin="lower",
    extent=(-80, 80, -80, 80),
    cmap="RdBu_r",
)
axes[0].set(
    title=r"$\mathrm{Re}\,\psi(\mathbf{x})$",
    xlabel=r"$x\;[h^{-1}\,\mathrm{Mpc}]$",
    ylabel=r"$y\;[h^{-1}\,\mathrm{Mpc}]$",
)
axes[1].imshow(
    fourier_wavelet[np.ix_(k_mask, k_mask)].T,
    origin="lower",
    extent=(-0.35, 0.35, -0.35, 0.35),
    cmap="magma",
)
axes[1].set(
    title=r"$|\widehat\psi(\mathbf{k})|$",
    xlabel=r"$k_x\;[h\,\mathrm{Mpc}^{-1}]$",
    ylabel=r"$k_y\;[h\,\mathrm{Mpc}^{-1}]$",
)
print(f"zero-mode amplitude: {abs(example_filter[0, 0]):.1e}")
savefig(fig, "02b_wavelet_localization.png")
plt.show()

## First-order scattering

Filtering gives a localized complex coefficient map. Taking its modulus and then averaging over the periodic box gives

$$
W_{j,\alpha}(\mathbf x)
=\mathrm{IFFT}\left[\widehat\delta\,\widehat\psi_{j,\alpha}\right],
\qquad
S_1(j,\alpha)=\left\langle|W_{j,\alpha}|\right\rangle_{\mathbf x}.
$$

The modulus is the nonlinear step that lets phase organization affect the summary. With four central wavenumbers and eight angles, the field is compressed to 32 numbers.

In [ ]:
def wavelet_response(field, wavelet_k):
    # TODO 2: convolve using multiplication in Fourier space.
    raise NotImplementedError


def first_order_coefficients(field):
    s1 = np.zeros((len(central_k), len(angles)))
    for j, k0 in enumerate(central_k):
        for a, angle in enumerate(angles):
            wavelet_k = morlet_filter(k0, angle)
            response = wavelet_response(field, wavelet_k)

            # TODO 3: take the modulus and then the spatial mean.
            s1[j, a] = np.nan
    return s1



s1 = first_order_coefficients(field)

response_small = wavelet_response(
    field, morlet_filter(central_k[1], 0.0)
)
response_large = wavelet_response(
    field, morlet_filter(central_k[2], np.pi / 4)
)

extent = (0, box_size, 0, box_size)
fig, axes = plt.subplots(
    2, 2, figsize=(9.4, 8.0), constrained_layout=True
)
axes[0, 0].imshow(
    field.T, origin="lower", extent=extent, cmap="RdBu_r"
)
axes[0, 0].set(
    title="Analytic field",
    xlabel=r"$x\;[h^{-1}\,\mathrm{Mpc}]$",
    ylabel=r"$y\;[h^{-1}\,\mathrm{Mpc}]$",
)
axes[0, 1].imshow(
    np.abs(response_small).T,
    origin="lower",
    extent=extent,
    cmap="viridis",
)
axes[0, 1].set(
    title=r"$|W|$: mode 8, $\alpha=0$",
    xlabel=r"$x\;[h^{-1}\,\mathrm{Mpc}]$",
    ylabel=r"$y\;[h^{-1}\,\mathrm{Mpc}]$",
)
axes[1, 0].imshow(
    np.abs(response_large).T,
    origin="lower",
    extent=extent,
    cmap="viridis",
)
axes[1, 0].set(
    title=r"$|W|$: mode 4, $\alpha=\pi/4$",
    xlabel=r"$x\;[h^{-1}\,\mathrm{Mpc}]$",
    ylabel=r"$y\;[h^{-1}\,\mathrm{Mpc}]$",
)
image = axes[1, 1].imshow(s1, origin="upper", aspect="auto", cmap="magma")
axes[1, 1].set(
    title=r"$S_1(j,\alpha)$",
    xlabel=r"wavevector angle $\alpha$ [deg]",
    ylabel="central Fourier mode",
    xticks=np.arange(8),
    xticklabels=[f"{angle:.1f}" for angle in np.degrees(angles)],
    yticks=np.arange(4),
    yticklabels=central_modes,
)
fig.colorbar(image, ax=axes[1, 1], label=r"$S_1$")
savefig(fig, "02b_wavelet_summary.png")
plt.show()

## What the average removes

Translating the field translates each response map. The spatial mean removes that absolute location, so $S_1$ is translation-invariant in this periodic example.

In [ ]:
shifted_field = np.roll(field, shift=(17, -11), axis=(0, 1))
shifted_s1 = first_order_coefficients(shifted_field)
fractional_change = np.max(np.abs(shifted_s1 - s1)) / np.max(s1)

print(f"largest fractional change in S1 after translation: {fractional_change:.2e}")
assert fractional_change < 1e-12

**Interpretation.** $S_1$ keeps scale and directional response but discards absolute position. For a real field, the channels at $\alpha$ and $\alpha+\pi$ are redundant after the modulus. Averaging over $\alpha$ would also remove anisotropy, which is often scientifically useful.

The lecture's second-order coefficient,

$$
S_2(\lambda_1,\lambda_2)
=
\left\langle
\left|
|\delta\star\psi_{\lambda_1}|\star\psi_{\lambda_2}
\right|
\right\rangle,
$$

asks whether fine-scale activity is modulated across a larger scale. We do not build the full second-order bank here.